<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/11_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# ==============================================================================
# 1. 패키지 설치 및 구글 드라이브 연결, 원자료 불러오기
# ==============================================================================

# 필요한 패키지 먼저 설치
!pip -q install pyreadstat

from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import pyreadstat
from IPython.display import display

# 데이터가 들어 있는 폴더 (구글 드라이브 루트 폴더로 수정)
DATA_DIR = '/content/drive/MyDrive'

# 파일 위치 자동 검색
file_path = os.path.join(DATA_DIR, 'hn24_all.sas7bdat')

if not os.path.exists(file_path):
    for root, dirs, files in os.walk(DATA_DIR):
        if 'hn24_all.sas7bdat' in files:
            file_path = os.path.join(root, 'hn24_all.sas7bdat')
            break

if not os.path.exists(file_path):
    raise FileNotFoundError(
        'hn24_all.sas7bdat 파일을 구글 드라이브에서 찾을 수 없습니다.'
    )

cols = [
    'age', 'sex', 'HE_DM_HbA1c', 'HE_HP', 'HE_obe',
    'incm', 'edu', 'BS3_1', 'BD1_11',
    'wt_itvex', 'kstrata', 'psu'
]

df, meta = pyreadstat.read_sas7bdat(
    file_path,
    usecols=cols
)

df.columns = df.columns.str.lower()

print('원자료 불러오기 완료')
print('파일 위치:', file_path)
print('원자료 크기:', df.shape)


# ==============================================================================
# 2. 변수 정의 및 전처리
# ==============================================================================

# 당뇨병: 0=비유병, 1=유병
df['dm'] = np.where(
    df['he_dm_hba1c'].isin([1, 2]), 0,
    np.where(df['he_dm_hba1c'] == 3, 1, np.nan)
)

# 고혈압: 0=비유병, 1=유병
df['hp'] = np.where(
    df['he_hp'].isin([1, 2, 3]), 0,
    np.where(df['he_hp'] == 4, 1, np.nan)
)

# 성별: 0=여자, 1=남자
df['sex_cat'] = np.where(
    df['sex'] == 2, 0,
    np.where(df['sex'] == 1, 1, np.nan)
)

# 연령: 만 19세 이상
df['age_num'] = np.where(
    df['age'] >= 19, df['age'], np.nan
)

# 비만: 0=비만 아님, 1=비만
df['obe_cat'] = np.where(
    df['he_obe'].isin([1, 2, 3]), 0,
    np.where(df['he_obe'].isin([4, 5, 6]), 1, np.nan)
)

# 소득수준
df['incm_cat'] = df['incm'].where(
    df['incm'].isin([1, 2, 3, 4]), np.nan
)

# 교육수준
df['edu_cat'] = df['edu'].where(
    df['edu'].isin([1, 2, 3, 4]), np.nan
)

# 흡연: 0=비흡연, 1=현재 흡연
df['smk_cat'] = np.where(
    df['bs3_1'].isin([3, 8]), 0,
    np.where(df['bs3_1'].isin([1, 2]), 1, np.nan)
)

# 음주: 0=월 1회 미만/안마심, 1=월 1회 이상
df['drk_cat'] = np.where(
    df['bd1_11'].isin([1, 2, 8]), 0,
    np.where(df['bd1_11'].isin([3, 4, 5, 6]), 1, np.nan)
)


# ==============================================================================
# 3. 연구 대상자 선정 및 모형별 분석 인원
# ==============================================================================

# Table 1 분석 대상자
sub_df = df[
    (df['age'] >= 19) &
    (df['dm'].notna()) &
    (df['hp'].notna())
].copy()

table1_n = len(sub_df)

# DAG 최소 보정집합
adjust_vars = [
    'age_num', 'sex_cat', 'incm_cat',
    'edu_cat', 'smk_cat', 'drk_cat', 'obe_cat'
]

# 모형 1: 노출변수와 결과변수 및 설계변수 결측 제거
model1_vars = [
    'dm', 'hp', 'wt_itvex', 'kstrata', 'psu'
]

model1_df = sub_df[model1_vars].dropna().copy()
model1_df = model1_df[model1_df['wt_itvex'] > 0].copy()

# 모형 2: DAG 보정변수까지 결측 제거
model2_vars = model1_vars + adjust_vars

model2_df = sub_df[model2_vars].dropna().copy()
model2_df = model2_df[model2_df['wt_itvex'] > 0].copy()

# 당뇨병 변수를 정수형으로 변환
model1_df['dm'] = model1_df['dm'].astype(int)
model2_df['dm'] = model2_df['dm'].astype(int)

print('=' * 60)
print('[1. 연구 대상자 및 모형별 분석 인원]')
print('=' * 60)

print('Table 1 분석 대상자:', table1_n, '명')
print('모형 1 분석 인원:', len(model1_df), '명')
print('모형 2 분석 인원:', len(model2_df), '명')


# ==============================================================================
# 4. 회귀분석 모형 설정
# ==============================================================================

# 모형 1: 무보정
formula1 = 'hp ~ C(dm, Treatment(reference=0))'

# 모형 2: DAG 기반 7개 변수 보정
formula2 = (
    'hp ~ C(dm, Treatment(reference=0))'
    ' + age_num + C(sex_cat) + C(incm_cat)'
    ' + C(edu_cat) + C(smk_cat)'
    ' + C(drk_cat) + C(obe_cat)'
)

# 모형 1 회귀분석
model1 = smf.glm(
    formula=formula1,
    data=model1_df,
    family=sm.families.Binomial(),
    freq_weights=model1_df['wt_itvex']
).fit(
    cov_type='cluster',
    cov_kwds={'groups': model1_df['psu']}
)

# 모형 2 회귀분석
model2 = smf.glm(
    formula=formula2,
    data=model2_df,
    family=sm.families.Binomial(),
    freq_weights=model2_df['wt_itvex']
).fit(
    cov_type='cluster',
    cov_kwds={'groups': model2_df['psu']}
)


# ==============================================================================
# 5. 오즈비, 95% 신뢰구간 및 p값 추출
# ==============================================================================

def get_dm_result(model, model_name, n):

    term = 'C(dm, Treatment(reference=0))[T.1]'

    coef = model.params[term]
    se = model.bse[term]

    OR = np.exp(coef)
    ci_lower = np.exp(coef - 1.96 * se)
    ci_upper = np.exp(coef + 1.96 * se)
    p_value = model.pvalues[term]

    return {
        '모형': model_name,
        'OR': OR,
        '95% CI': f'{ci_lower:.3f} - {ci_upper:.3f}',
        'p값': p_value,
        '분석 인원': n
    }


# 모형별 결과 추출
result1 = get_dm_result(
    model1, '모형 1 (무보정)', len(model1_df)
)

result2 = get_dm_result(
    model2, '모형 2 (보정)', len(model2_df)
)


# ==============================================================================
# 6. 최종 Table 2 및 분석 인원 비교 출력
# ==============================================================================

# Table 2 작성
table2 = pd.DataFrame([
    {
        '노출변수': '당뇨병 비유병 (기준군)',
        '모형 1 OR (95% CI)': '1.00 (Ref.)',
        '모형 1 p값': '-',
        '모형 2 OR (95% CI)': '1.00 (Ref.)',
        '모형 2 p값': '-'
    },
    {
        '노출변수': '당뇨병 유병',
        '모형 1 OR (95% CI)': (
            f"{result1['OR']:.3f} ({result1['95% CI']})"
        ),
        '모형 1 p값': f"{result1['p값']:.4f}",
        '모형 2 OR (95% CI)': (
            f"{result2['OR']:.3f} ({result2['95% CI']})"
        ),
        '모형 2 p값': f"{result2['p값']:.4f}"
    }
])

print('\n' + '=' * 60)
print('[2. C-2. TABLE 2 회귀분석 결과]')
print('=' * 60)

display(table2)


# 모형별 분석 인원 및 Table 1과의 차이
count_result = pd.DataFrame({
    '구분': ['Table 1', '모형 1', '모형 2'],
    '분석 인원': [
        table1_n,
        len(model1_df),
        len(model2_df)
    ],
    'Table 1과의 차이': [
        0,
        table1_n - len(model1_df),
        table1_n - len(model2_df)
    ]
})

print('\n' + '=' * 60)
print('[3. Table 1과 모형별 분석 인원 비교]')
print('=' * 60)

display(count_result)


# 분석 인원 차이 확인
print('\n' + '=' * 60)
print('[4. 최종 분석 결과 확인]')
print('=' * 60)

print('Table 1 분석 대상자:', table1_n, '명')
print('모형 1 분석 인원:', len(model1_df), '명')
print('모형 2 분석 인원:', len(model2_df), '명')

print(
    '모형 1 제외 인원:',
    table1_n - len(model1_df), '명'
)

print(
    '모형 2 제외 인원:',
    table1_n - len(model2_df), '명'
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
원자료 불러오기 완료
파일 위치: /content/drive/MyDrive/hn24_all.sas7bdat
원자료 크기: (6997, 12)
[1. 연구 대상자 및 모형별 분석 인원]
Table 1 분석 대상자: 5567 명
모형 1 분석 인원: 5567 명
모형 2 분석 인원: 5367 명


/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(
/usr/local/lib/python3.13/dist-packages/statsmodels/genmod/generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(



[2. C-2. TABLE 2 회귀분석 결과]


,노출변수,모형 1 OR (95% CI),모형 1 p값,모형 2 OR (95% CI),모형 2 p값
0,당뇨병 비유병 (기준군),1.00 (Ref.),-,1.00 (Ref.),-
1,당뇨병 유병,5.466 (4.560 - 6.552),0.0000,2.356 (1.915 - 2.898),0.0000



[3. Table 1과 모형별 분석 인원 비교]


,구분,분석 인원,Table 1과의 차이
0,Table 1,5567,0
1,모형 1,5567,0
2,모형 2,5367,200



[4. 최종 분석 결과 확인]
Table 1 분석 대상자: 5567 명
모형 1 분석 인원: 5567 명
모형 2 분석 인원: 5367 명
모형 1 제외 인원: 0 명
모형 2 제외 인원: 200 명
